# 골전도+공기전도 융합 음성 향상 — 훈련 노트북

**실험 설계서 v3 기준**

## Drive 전략 (Google 권장)
```
Drive ↔ Colab 전송 = 단일 tar.gz 파일만
  - 원본 DB(LibriSpeech 등): /content/ 직접 다운로드 (Drive 거치지 않음)
  - processed 데이터: Drive의 tar.gz → 로컬 압축 해제
  - 결과/체크포인트: 로컬 압축 → Drive 단일 파일
```

> **GPU 런타임** 필수: 런타임 → 런타임 유형 변경 → A100 / T4

---
## 0. 환경 확인

In [ ]:
import torch, os
print(f'PyTorch: {torch.__version__}')
print(f'GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else "없음"}')
if torch.cuda.is_available():
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory/1e9:.1f}GB')
!df -h /content | tail -1
!nvidia-smi --query-gpu=name,memory.free --format=csv,noheader 2>/dev/null || true

---
## 1. 패키지 설치 & 코드 클론

In [ ]:
%%capture
!pip install soundfile librosa pesq pystoi onnx onnxruntime pyyaml tqdm seaborn

In [ ]:
REPO_URL = 'https://github.com/sungmin-park-dev/tactical-speech-enhancement.git'
REPO_DIR = '/content/tactical-speech-enhancement'
if not os.path.exists(REPO_DIR):
    !git clone {REPO_URL} {REPO_DIR}
else:
    !cd {REPO_DIR} && git pull --quiet
import sys
sys.path.insert(0, REPO_DIR)
print('코드 준비 완료')

---
## 2. 데이터 준비

### 경우 A: **최초 실행** → `download_to_drive.ipynb` 를 먼저 실행  
### 경우 B: **재사용** → 아래 셀에서 Drive tar.gz → 로컬 복원

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

DRIVE_ROOT   = '/content/drive/MyDrive/tactical-speech-enhancement'
DRIVE_ARCHIVE = f'{DRIVE_ROOT}/data/processed_data.tar.gz'

if os.path.exists('/content/data/processed'):
    print('processed 데이터 이미 로컬에 있음 — 스킵')
elif os.path.exists(DRIVE_ARCHIVE):
    # Google 권장: 단일 tar.gz 복사 → 로컬 압축 해제
    print('Drive에서 processed 데이터 복원 중...')
    !cp {DRIVE_ARCHIVE} /content/processed_data.tar.gz
    !mkdir -p /content/data && tar -xzf /content/processed_data.tar.gz -C /content/data/
    !rm /content/processed_data.tar.gz
    print('복원 완료')
else:
    print('⚠ processed 데이터 없음!')
    print('download_to_drive.ipynb 를 먼저 실행하세요.')

!du -sh /content/data/processed/* 2>/dev/null || true

In [ ]:
# Colab config 설정 (processed 경로)
import yaml
with open(f'{REPO_DIR}/configs/data_config.yaml') as f:
    cfg = yaml.safe_load(f)
cfg['paths']['output_root'] = '/content/data/processed'
COLAB_CFG = '/content/data_config_colab.yaml'
with open(COLAB_CFG, 'w') as f:
    yaml.dump(cfg, f, allow_unicode=True)
print('config 준비 완료:', COLAB_CFG)

---
## 3. 모델 훈련

> D3 (DPCRN) 구현 완료 후 주석 해제

In [ ]:
# ── 훈련 설정 ──
ABLATION  = 'A-soft'    # A-soft | A-hard | A-param | B | C
BACKBONE  = 'dpcrn'     # dpcrn | unet
ENV       = 'military'  # military | general | mixed
SEED      = 0
EPOCHS    = 50
BATCH     = 16          # A100: 16 / T4: 8
MASK_TYPE = 'soft'      # soft | hard | parametric
ALPHA     = 0.5         # MR-STFT 가중치

SAVE_DIR = f'/content/results/{BACKBONE}/{ABLATION}/seed{SEED}'
print(f'설정: {BACKBONE}/{ABLATION}/env={ENV}/seed={SEED}')
print(f'저장: {SAVE_DIR}')

In [ ]:
# D3 구현 완료 후 주석 해제
# !python {REPO_DIR}/train.py \
#     --config {COLAB_CFG} \
#     --backbone {BACKBONE} \
#     --ablation {ABLATION} \
#     --env {ENV} \
#     --mask_type {MASK_TYPE} \
#     --alpha {ALPHA} \
#     --seed {SEED} \
#     --epochs {EPOCHS} \
#     --batch_size {BATCH} \
#     --save_dir {SAVE_DIR}
print('train.py (D3) 준비 후 위 주석 해제')

---
## 4. 결과 Drive 저장

> Google 권장: 단일 tar.gz로 압축 후 Drive 복사 (I/O 할당량 최소화)

In [ ]:
import time
DRIVE_RESULTS = f'{DRIVE_ROOT}/results'
!mkdir -p {DRIVE_RESULTS}

if os.path.exists('/content/results'):
    TS = int(time.time())
    LOCAL_ARCHIVE = f'/content/results_{TS}.tar.gz'
    print('결과 압축 중...')
    !tar -czf {LOCAL_ARCHIVE} -C /content results
    !du -sh {LOCAL_ARCHIVE}
    print('Drive로 복사 중... (단일 파일)')
    !cp {LOCAL_ARCHIVE} {DRIVE_RESULTS}/results_{TS}.tar.gz
    !rm {LOCAL_ARCHIVE}
    print(f'Drive 저장 완료: {DRIVE_RESULTS}/results_{TS}.tar.gz')
else:
    print('저장할 결과 없음')

---
## [선택] 파이프라인 빠른 검증

In [ ]:
import numpy as np
from data.bc_simulator import BCSimulator
from data.saturation import SaturationSimulator
from data.noise_mixer import NoiseMixer
from data.pipeline import SampleGenerator

FS = 16000
rng = np.random.default_rng(0)
t = np.linspace(0, 4, FS*4, endpoint=False)
speech = (0.3 * np.sin(2*np.pi*200*t)).astype('float32')

bc_sim  = BCSimulator(sample_rate=FS, rng=rng)
sat_sim = SaturationSimulator(sample_rate=FS, rng=rng)
n_mixer = NoiseMixer(env='military', sample_rate=FS, rng=rng)
gen = SampleGenerator('military', bc_sim, sat_sim, n_mixer, rng=rng)

sample = gen.generate(speech)
print('bc_signal:', sample['bc_signal'].shape)
print('ac_noisy: ', sample['ac_noisy'].shape)
print('masks:', {k: v.shape for k, v in sample['masks'].items()})
print('\n파이프라인 검증 통과!')